# **Read Bronze datasets**

In [1]:
crm_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Data/crm_customer_data.csv")
)

service_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Data/service_provisioning_data.csv")
)

billing_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Data/billing_usage_data.csv")
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 3, Finished, Available, Finished, False)

In [2]:
print("CRM Rows:", crm_df.count())
print("Service Rows:", service_df.count())
print("Billing Rows:", billing_df.count())

display(crm_df.limit(5))
display(service_df.limit(5))
display(billing_df.limit(5))

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 4, Finished, Available, Finished, False)

CRM Rows: 7070
Service Rows: 7043
Billing Rows: 7071


SynapseWidget(Synapse.DataFrame, 43a337b5-4779-4884-8589-526a5a1af504)

SynapseWidget(Synapse.DataFrame, 8aae2762-52e0-4b6f-9e78-a1aca3611d3a)

SynapseWidget(Synapse.DataFrame, d7c3bfa7-144e-4b7c-95cf-2b0fde52700e)

# **Check Schema** 

In [3]:
crm_df.printSchema()
service_df.printSchema()
billing_df.printSchema()

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 5, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- senior_citizen: string (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependent: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- paperless_billing: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- lattitude: double (nullable = true)
 |-- longitude: double (nullable = true)

root
 |-- CustomerID: string (nullable = true)
 |-- Tenure Months: integer (nullable = true)
 |-- Phone Service: string (nullable = true)
 |-- Multiple Lines: string (nullable = true)
 |-- Internet Service: string (nullable = true)
 |-- Online Security: string (nullable = true)
 |-- Online Backup: string (nullable = true)
 |-- Device Protection: string (nullable = true)
 |-- Tech Support: string (nullable = true)


# **Schema Drift Detection**

Validate incoming Bronze schemas against the expected source schemas before Silver transformation.

In [4]:
expected_crm_columns = {
    "customer_id",
    "gender",
    "senior_citizen",
    "partner",
    "dependent",
    "contract",
    "paperless_billing",
    "payment_method",
    "country",
    "state",
    "city",
    "zip_code",
    "lattitude",
    "longitude"
}

expected_service_columns = {
    "CustomerID",
    "Tenure Months",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies"
}

expected_billing_columns = {
    "Customer_ID",
    "Monthly_Charges",
    "Total_Charges",
    "Churn_Label",
    "Churn_Reason",
    "Churn_Value",
    "Churn_Score"
}

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 6, Finished, Available, Finished, False)

In [5]:
def detect_schema_drift(df, expected_columns, source_name):

    actual_columns = set(df.columns)

    missing_columns = expected_columns - actual_columns
    new_columns = actual_columns - expected_columns

    print(f"\nSchema Validation: {source_name}")
    print("-" * 50)

    if missing_columns:
        print("Missing Columns:", missing_columns)
    else:
        print("No missing columns")

    if new_columns:
        print("New / Unexpected Columns:", new_columns)
    else:
        print("No unexpected columns")

    if not missing_columns and not new_columns:
        print("Schema validation passed")

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 7, Finished, Available, Finished, False)

In [6]:
detect_schema_drift(
    crm_df,
    expected_crm_columns,
    "CRM Customer Data"
)

detect_schema_drift(
    service_df,
    expected_service_columns,
    "Service Provisioning Data"
)

detect_schema_drift(
    billing_df,
    expected_billing_columns,
    "Billing and Usage Data"
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 8, Finished, Available, Finished, False)


Schema Validation: CRM Customer Data
--------------------------------------------------
No missing columns
No unexpected columns
Schema validation passed

Schema Validation: Service Provisioning Data
--------------------------------------------------
No missing columns
No unexpected columns
Schema validation passed

Schema Validation: Billing and Usage Data
--------------------------------------------------
Missing Columns: {'Churn_Score'}
No unexpected columns


# **Standardize schemas**

In [7]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 9, Finished, Available, Finished, False)

# **Column name correction**

In [8]:
def standardize_column_name(column_name):
    """
    Convert column names to lowercase snake_case.
    Customer_ID     -> customer_id
    """

    name = column_name.strip()

    # Handle known customer ID variations
    if name.lower().replace("_", "").replace(" ", "") == "customerid":
        return "customer_id"

    # Insert underscore between lowercase and uppercase letters
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)

    # Replace spaces and special characters with underscores
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)

    # Remove repeated or leading/trailing underscores
    name = re.sub(r"_+", "_", name).strip("_")

    return name.lower()

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 10, Finished, Available, Finished, False)

In [9]:
def standardize_dataframe_columns(df):
    for old_column in df.columns:
        new_column = standardize_column_name(old_column)

        if old_column != new_column:
            df = df.withColumnRenamed(old_column, new_column)

    return df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 11, Finished, Available, Finished, False)

In [10]:
crm_silver_df = standardize_dataframe_columns(crm_df)
service_silver_df = standardize_dataframe_columns(service_df)
billing_silver_df = standardize_dataframe_columns(billing_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 12, Finished, Available, Finished, False)

In [11]:
print("CRM columns:")
print(crm_silver_df.columns)

print("\nService columns:")
print(service_silver_df.columns)

print("\nBilling columns:")
print(billing_silver_df.columns)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 13, Finished, Available, Finished, False)

CRM columns:
['customer_id', 'gender', 'senior_citizen', 'partner', 'dependent', 'contract', 'paperless_billing', 'payment_method', 'country', 'state', 'city', 'zip_code', 'lattitude', 'longitude']

Service columns:
['customer_id', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies']

Billing columns:
['customer_id', 'monthly_charges', 'total_charges', 'churn_label', 'churn_reason', 'churn_value']


In [12]:
#The customer key should now be identical in every dataset

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 14, Finished, Available, Finished, False)

****

# **Clean string columns**

In [13]:
def clean_string_columns(df):
    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    for column_name in string_columns:
        df = df.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

        df = df.withColumn(
            column_name,
            F.when(
                F.lower(F.col(column_name)).isin(
                    "", "null", "none", "n/a", "na", "unknown"
                ),
                F.lit(None)
            ).otherwise(F.col(column_name))
        )

    return df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 15, Finished, Available, Finished, False)

In [14]:
crm_silver_df = clean_string_columns(crm_silver_df)
service_silver_df = clean_string_columns(service_silver_df)
billing_silver_df = clean_string_columns(billing_silver_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 16, Finished, Available, Finished, False)

# **Correct data types**

In [15]:
billing_silver_df = (
    billing_silver_df
    .withColumn(
        "monthly_charges",
        F.col("monthly_charges").cast(DoubleType())
    )
    .withColumn(
        "total_charges",
        F.when(
            F.trim(F.col("total_charges")) == "",
            None
        ).otherwise(
            F.regexp_replace(
                F.col("total_charges"),
                ",",
                ""
            ).cast(DoubleType())
        )
    )
    .withColumn(
        "churn_value",
        F.col("churn_value").cast(IntegerType())
    )
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 17, Finished, Available, Finished, False)

In [16]:
crm_silver_df = (
    crm_silver_df
    .withColumn(
        "zip_code",
        F.col("zip_code").cast("string")
    )
    .withColumn(
        "lattitude",
        F.col("lattitude").cast(DoubleType())
    )
    .withColumnRenamed("lattitude", "latitude")
    .withColumn(
        "longitude",
        F.col("longitude").cast(DoubleType())
    )
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 18, Finished, Available, Finished, False)

In [17]:
service_silver_df = service_silver_df.withColumn(
    "tenure_months",
    F.col("tenure_months").cast(IntegerType())
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 19, Finished, Available, Finished, False)

In [18]:
crm_silver_df.printSchema()
service_silver_df.printSchema()
billing_silver_df.printSchema()

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 20, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- senior_citizen: string (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependent: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- paperless_billing: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- phone_service: string (nullable = true)
 |-- multiple_lines: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- online_security: string (nullable = true)
 |-- online_backup: string (nullable = true)
 |-- device_protection: string (nullable = true)
 |-- tech_support: string (nullable = true)
 

# **Data-quality profiling function**

In [19]:
def profile_dataframe(df, dataset_name, key_column="customer_id"):

    print("=" * 70)
    print(f"DATA QUALITY REPORT: {dataset_name}")
    print("=" * 70)

    row_count = df.count()
    column_count = len(df.columns)

    print(f"Row count: {row_count}")
    print(f"Column count: {column_count}")

    if key_column in df.columns:
        unique_key_count = (
            df.select(key_column)
              .distinct()
              .count()
        )

        null_key_count = (
            df.filter(F.col(key_column).isNull())
              .count()
        )

        duplicate_key_count = (
            df.groupBy(key_column)
              .count()
              .filter(
                  (F.col(key_column).isNotNull()) &
                  (F.col("count") > 1)
              )
              .count()
        )

        print(f"Unique customer IDs: {unique_key_count}")
        print(f"Null customer IDs: {null_key_count}")
        print(f"Duplicated customer IDs: {duplicate_key_count}")

    null_summary = df.select([
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ).alias(column_name)
        for column_name in df.columns
    ])

    print("\nNull counts:")
    display(null_summary)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 21, Finished, Available, Finished, False)

In [20]:
profile_dataframe(
    crm_silver_df,
    "CRM Customer Data"
)

profile_dataframe(
    service_silver_df,
    "Service Provisioning Data"
)

profile_dataframe(
    billing_silver_df,
    "Billing and Churn Data"
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 22, Finished, Available, Finished, False)

DATA QUALITY REPORT: CRM Customer Data
Row count: 7070
Column count: 14
Unique customer IDs: 7043
Null customer IDs: 0
Duplicated customer IDs: 27

Null counts:


SynapseWidget(Synapse.DataFrame, 42be5bda-7598-4fbb-87cf-9020faa70973)

DATA QUALITY REPORT: Service Provisioning Data
Row count: 7043
Column count: 11
Unique customer IDs: 7043
Null customer IDs: 0
Duplicated customer IDs: 0

Null counts:


SynapseWidget(Synapse.DataFrame, 72617ba5-ade6-4e42-af69-c090cdb6c3e1)

DATA QUALITY REPORT: Billing and Churn Data
Row count: 7071
Column count: 6
Unique customer IDs: 7043
Null customer IDs: 0
Duplicated customer IDs: 28

Null counts:


SynapseWidget(Synapse.DataFrame, 6262fabe-c247-4f7f-af01-42c135ad05bf)

# **Inspect duplicate records**

In [21]:
def show_duplicate_customers(df, dataset_name):
    duplicates = (
        df.groupBy("customer_id")
          .count()
          .filter(
              (F.col("customer_id").isNotNull()) &
              (F.col("count") > 1)
          )
    )

    print(f"Duplicate customers in {dataset_name}:")
    display(duplicates)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 23, Finished, Available, Finished, False)

In [22]:
show_duplicate_customers(
    crm_silver_df,
    "CRM"
)

show_duplicate_customers(
    service_silver_df,
    "Service"
)

show_duplicate_customers(
    billing_silver_df,
    "Billing"
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 24, Finished, Available, Finished, False)

Duplicate customers in CRM:


SynapseWidget(Synapse.DataFrame, c3ff8ffd-1169-4773-9d76-d539ed3b322c)

Duplicate customers in Service:


SynapseWidget(Synapse.DataFrame, c6069b19-ef33-4bcd-96f2-5798484bcadb)

Duplicate customers in Billing:


SynapseWidget(Synapse.DataFrame, 652dac71-848e-49d2-98f8-4e08f4a5785a)

# **Drop duplicates**

In [23]:
crm_silver_df = crm_silver_df.dropDuplicates(["customer_id"])
service_silver_df = service_silver_df.dropDuplicates(["customer_id"])
billing_silver_df = billing_silver_df.dropDuplicates(["customer_id"])

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 25, Finished, Available, Finished, False)

# **Remove invalid customer-key records**

In [24]:
crm_silver_df = crm_silver_df.filter(
    F.col("customer_id").isNotNull()
)

service_silver_df = service_silver_df.filter(
    F.col("customer_id").isNotNull()
)

billing_silver_df = billing_silver_df.filter(
    F.col("customer_id").isNotNull()
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 26, Finished, Available, Finished, False)

In [25]:
crm_silver_df = crm_silver_df.withColumn(
    "customer_id",
    F.upper(F.trim(F.col("customer_id")))
)

service_silver_df = service_silver_df.withColumn(
    "customer_id",
    F.upper(F.trim(F.col("customer_id")))
)

billing_silver_df = billing_silver_df.withColumn(
    "customer_id",
    F.upper(F.trim(F.col("customer_id")))
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 27, Finished, Available, Finished, False)

# **Domain Validation**

In [26]:
from pyspark.sql.types import StringType


def inspect_string_domains(df, exclude_columns=["customer_id","zip_code", "country", "churn_reason", "state", "city"]):
    """
    Inspect distinct values and their frequencies
    for every string column.
    """

    string_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
        and field.name not in exclude_columns
    ]

    print("=" * 80)
    print("STRING DOMAIN VALIDATION")
    print("=" * 80)

    for column_name in string_columns:

        print(f"\n{'='*60}")
        print(f"Column : {column_name}")
        print(f"{'='*60}")

        summary_df = (
            df.groupBy(column_name)
              .count()
              .orderBy(F.desc("count"))
        )

        print(f"Distinct Values : {summary_df.count()}")

        display(summary_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 28, Finished, Available, Finished, False)

In [27]:
inspect_string_domains(crm_silver_df)

inspect_string_domains(service_silver_df)

inspect_string_domains(billing_silver_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 29, Finished, Available, Finished, False)

STRING DOMAIN VALIDATION

Column : gender
Distinct Values : 11


SynapseWidget(Synapse.DataFrame, 44f62d61-0068-4c51-a1e1-5136df09bc4b)


Column : senior_citizen
Distinct Values : 2


SynapseWidget(Synapse.DataFrame, 24505fdc-d8a1-4239-8ac0-486f626194ef)

...

SynapseWidget(Synapse.DataFrame, ee4b4329-61fd-4449-82c5-3ecf66b9307b)


Column : streaming_movies
Distinct Values : 3


SynapseWidget(Synapse.DataFrame, ab78f4ad-0d91-40c4-ad53-f46db0680200)

STRING DOMAIN VALIDATION

Column : churn_label
Distinct Values : 2


SynapseWidget(Synapse.DataFrame, 71be0112-a8de-4a35-8836-8cea3830ee60)

In [28]:
crm_silver_df = crm_silver_df.withColumn(
    "gender",
    F.when(
        F.lower(F.trim(F.col("gender"))).isin("male", "m"),
        "Male"
    )
    .when(
        F.lower(F.trim(F.col("gender"))).isin(
            "female",
            "f",
            "femlae"
        ),
        "Female"
    )
    .otherwise(F.lit(None)))


StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 30, Finished, Available, Finished, False)

In [29]:
service_silver_df = service_silver_df.withColumn(
    "internet_service",
    F.when(
        F.lower(F.trim(F.col("internet_service"))).isin("fiber optic"),
        "Fiber Optic"
    )
    .when(
        F.lower(F.trim(F.col("internet_service"))).isin(
            "dsl"
        ),
        "DSL"
    )
    .when(
        F.lower(F.trim(F.col("internet_service"))).isin(
            "no"
        ),
        "No"
    )
    .otherwise(F.lit(None)))


StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 31, Finished, Available, Finished, False)

In [30]:
inspect_string_domains(crm_silver_df)

inspect_string_domains(service_silver_df)

inspect_string_domains(billing_silver_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 32, Finished, Available, Finished, False)

STRING DOMAIN VALIDATION

Column : gender
Distinct Values : 2


SynapseWidget(Synapse.DataFrame, dbc4c0b0-325b-4772-83b1-524fc2189c5e)


Column : senior_citizen
Distinct Values : 2


SynapseWidget(Synapse.DataFrame, 1410f4d5-1a7e-4eb4-a755-27ebdd2ace31)

...

SynapseWidget(Synapse.DataFrame, 6d4d0524-4da2-4360-b373-ed44a32d2ecd)


Column : streaming_movies
Distinct Values : 3


SynapseWidget(Synapse.DataFrame, 8443bab5-2b70-4a93-81f8-f35639631ebb)

STRING DOMAIN VALIDATION

Column : churn_label
Distinct Values : 2


SynapseWidget(Synapse.DataFrame, 8218f4bb-53ec-486b-a55f-f6216e0308ff)

# **Validate business rules**

In [31]:
from pyspark.sql import functions as F


def validate_business_rules(
    crm_df,
    service_df,
    billing_df,
    display_results=True
):
    """
    Validate Silver-layer business rules.

    Returns
    -------
    result_df : Summary containing each validation rule and failed record count.

    failed_records : Dictionary containing the actual failed-record DataFrame
        for each validation rule.
    """

    
    # 1. Define the failed records for each validation rule
    

    failed_records = {
        "negative_tenure": service_df.filter(
            F.col("tenure_months") < 0
        ),

        "negative_monthly_charges": billing_df.filter(
            F.col("monthly_charges") < 0
        ),

        "negative_total_charges": billing_df.filter(
            F.col("total_charges") < 0
        ),

        "invalid_churn_value": billing_df.filter(
            F.col("churn_value").isNull()
            | ~F.col("churn_value").isin(0, 1)
        )
    }

    
    # 2. Count failed records for each rule
    

    checks = {
        rule_name: failed_df.count()
        for rule_name, failed_df in failed_records.items()
    }

    
    # 3. Convert the results into a Spark DataFrame
    

    result_rows = [
        (
            rule_name,
            failed_count,
            "PASS" if failed_count == 0 else "FAIL"
        )
        for rule_name, failed_count in checks.items()
    ]

    result_df = spark.createDataFrame(
        result_rows,
        [
            "validation_rule",
            "failed_records",
            "validation_status"
        ]
    )

    
    # 4. Display summary and failed records
    

    if display_results:

        print("Business rule validation summary")
        display(result_df)

        for rule_name, failed_df in failed_records.items():

            failed_count = checks[rule_name]

            if failed_count > 0:
                print(
                    f"Rule failed: {rule_name} "
                    f"({failed_count} records)"
                )

                display(failed_df)

    
    # 5. Return both summary and failed records
    

    return result_df, failed_records

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 33, Finished, Available, Finished, False)

In [32]:
validation_summary_df = validate_business_rules(
    crm_silver_df,
    service_silver_df,
    billing_silver_df
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 34, Finished, Available, Finished, False)

Business rule validation summary


SynapseWidget(Synapse.DataFrame, 067dbece-9e14-4d63-8e3a-954a9deca508)

Rule failed: negative_monthly_charges (2 records)


SynapseWidget(Synapse.DataFrame, 47ac0f98-9e5d-4c94-8398-d124ceffac7e)

Rule failed: negative_total_charges (1 records)


SynapseWidget(Synapse.DataFrame, e7e7568e-3399-4e70-9af3-037473e911a6)

# **Making Quarantine Tables**

In [65]:
service_important_columns = [
    "customer_id",
    "tenure_months",
    "paperless_billing",
    "payment_method",
    "online_security",
    "tech_support",
    "internet_service"
]

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 74, Finished, Available, Finished, False)

In [66]:
def split_service_valid_and_quarantine(service_df):

    negative_tenure = (
        F.col("tenure_months").isNotNull()
        & (F.col("tenure_months") < 0)
    )

    missing_customer_id = F.col("customer_id").isNull()
    missing_tenure = F.col("tenure_months").isNull()
    missing_online_security = F.col("online_security").isNull()
    missing_tech_support = F.col("tech_support").isNull()
    missing_internet_service = F.col("internet_service").isNull()



    quarantine_condition = (
        missing_customer_id
        | missing_tenure
        | negative_tenure
        | missing_online_security
        | missing_tech_support
        | missing_internet_service
    )

    reason_array = F.array(
        F.when(
            missing_customer_id,
            F.lit("missing_customer_id")
        ),
        F.when(
            missing_tenure,
            F.lit("missing_tenure_months")
        ),
        F.when(
            negative_tenure,
            F.lit("negative_tenure")
        ),
        F.when(
            missing_online_security,
            F.lit("missing_online_security")
        ),
        F.when(
            missing_tech_support,
            F.lit("missing_tech_support")
        ),
        F.when(
            missing_internet_service,
            F.lit("missing_internet_service")
        )
    )

    quarantine_df = (
        service_df
        .filter(quarantine_condition)
        .withColumn(
            "quarantine_reasons",
            F.filter(
                reason_array,
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )
    )

    valid_df = service_df.filter(
        ~quarantine_condition
    )

    return valid_df, quarantine_df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 75, Finished, Available, Finished, False)

In [67]:
service_silver, service_quarantine_df = (
    split_service_valid_and_quarantine(
        service_silver_df
    )
)

display(service_quarantine_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 76, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46b07637-fc05-41b4-b1b2-2736a6ef0011)

In [68]:
def split_billing_valid_and_quarantine(billing_df):

    missing_customer_id = F.col("customer_id").isNull()
    missing_monthly_charges = F.col("monthly_charges").isNull()
    missing_total_charges = F.col("total_charges").isNull()
    missing_churn_value = F.col("churn_value").isNull()

    negative_monthly_charges = (
        F.col("monthly_charges").isNotNull()
        & (F.col("monthly_charges") < 0)
    )

    negative_total_charges = (
        F.col("total_charges").isNotNull()
        & (F.col("total_charges") < 0)
    )

    invalid_churn_value = (
        F.col("churn_value").isNotNull()
        & ~F.col("churn_value").isin(0, 1)
    )


    quarantine_condition = (
        missing_customer_id
        | missing_monthly_charges
        | missing_total_charges
        | missing_churn_value
        | negative_monthly_charges
        | negative_total_charges
        | invalid_churn_value
    )

    reason_array = F.array(
        F.when(
            missing_customer_id,
            F.lit("missing_customer_id")
        ),
        F.when(
            missing_monthly_charges,
            F.lit("missing_monthly_charges")
        ),
        F.when(
            missing_total_charges,
            F.lit("missing_total_charges")
        ),
        F.when(
            missing_churn_value,
            F.lit("missing_churn_value")
        ),
        F.when(
            negative_monthly_charges,
            F.lit("negative_monthly_charges")
        ),
        F.when(
            negative_total_charges,
            F.lit("negative_total_charges")
        ),
        F.when(
            invalid_churn_value,
            F.lit("invalid_churn_value")
        )
    )

    quarantine_df = (
        billing_df
        .filter(quarantine_condition)
        .withColumn(
            "quarantine_reasons",
            F.filter(
                reason_array,
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )
    )

    valid_df = billing_df.filter(
        ~quarantine_condition
    )

    return valid_df, quarantine_df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 77, Finished, Available, Finished, False)

In [69]:
billing_silver, billing_quarantine_df = (
    split_billing_valid_and_quarantine(
        billing_silver_df
    )
)

display(billing_quarantine_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 328e6594-5312-4dd6-a0ce-36917bb6afc8)

In [70]:
def split_crm_valid_and_quarantine(crm_df):

    missing_customer_id = F.col("customer_id").isNull()
    missing_gender = F.col("gender").isNull()
    missing_paperless_billing = F.col("paperless_billing").isNull()
    missing_payment_method = F.col("payment_method").isNull()

    invalid_gender = (
        F.col("gender").isNotNull()
        & ~F.col("gender").isin("Male", "Female")
    )

    quarantine_condition = (
        missing_customer_id
        | missing_gender
        | invalid_gender
        | missing_paperless_billing
        | missing_payment_method
        
    )

    reason_array = F.array(
        F.when(
            missing_customer_id,
            F.lit("missing_customer_id")
        ),
        F.when(
            missing_gender,
            F.lit("missing_gender")
        ),
        F.when(
            invalid_gender,
            F.lit("invalid_gender")
        ),
        F.when(
            missing_paperless_billing,
            F.lit("missing_paperless_billing")
        ),
        F.when(
            missing_payment_method,
            F.lit("missing_payment_method")
        )
    )

    quarantine_df = (
        crm_df
        .filter(quarantine_condition)
        .withColumn(
            "quarantine_reasons",
            F.filter(
                reason_array,
                lambda reason: reason.isNotNull()
            )
        )
        .withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )
    )

    valid_df = crm_df.filter(
        ~quarantine_condition
    )

    return valid_df, quarantine_df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 79, Finished, Available, Finished, False)

In [71]:
crm_silver, crm_quarantine_df = (
    split_crm_valid_and_quarantine(
        crm_silver_df
    )
)

display(crm_quarantine_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 80, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3f3a2e47-07ac-4c40-bf5b-42f9ea1de6a3)

# **Adding qurantine records for all three tables**

In [72]:
all_quarantine_ids_df = (
    crm_quarantine_df
    .select("customer_id")

    .unionByName(
        billing_quarantine_df.select("customer_id")
    )

    .unionByName(
        service_quarantine_df.select("customer_id")
    )

    .filter(F.col("customer_id").isNotNull())
    .distinct()
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 81, Finished, Available, Finished, False)

In [73]:
print(
    "Customers quarantined in at least one domain:",
    all_quarantine_ids_df.count()
)

display(all_quarantine_ids_df)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 82, Finished, Available, Finished, False)

Customers quarantined in at least one domain: 186


SynapseWidget(Synapse.DataFrame, 23759816-85d0-44bf-b1b7-adcaff6bb811)

In [74]:
def synchronize_domain_quarantine(
    silver_df,
    quarantine_df,
    all_quarantine_ids_df,
    domain_name
):
    # Valid rows that must also be quarantined because
    # the customer failed validation in another domain
    related_quarantine_rows_df = (
        silver_df
        .join(
            all_quarantine_ids_df,
            on="customer_id",
            how="left_semi"
        )
        .withColumn(
            "quarantine_reason",
            F.array(
                F.lit(
                    "customer_failed_validation_in_another_domain"
                )
            )
        )
        .withColumn(
            "quarantine_source",
            F.lit(domain_name)
        )
        .withColumn(
            "quarantine_timestamp",
            F.current_timestamp()
        )
    )

    # Add related records to the existing quarantine
    updated_quarantine_df = (
        quarantine_df
        .unionByName(
            related_quarantine_rows_df,
            allowMissingColumns=True
        )
        .dropDuplicates(["customer_id"])
    )

    # Keep only customers that are valid across every domain
    updated_silver_df = (
        silver_df
        .join(
            all_quarantine_ids_df,
            on="customer_id",
            how="left_anti"
        )
    )

    return updated_silver_df, updated_quarantine_df

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 83, Finished, Available, Finished, False)

In [75]:
crm_silver, crm_quarantine_df = (
    synchronize_domain_quarantine(
        silver_df=crm_silver,
        quarantine_df=crm_quarantine_df,
        all_quarantine_ids_df=all_quarantine_ids_df,
        domain_name="crm_customer"
    )
)

billing_silver, billing_quarantine_df = (
    synchronize_domain_quarantine(
        silver_df=billing_silver,
        quarantine_df=billing_quarantine_df,
        all_quarantine_ids_df=all_quarantine_ids_df,
        domain_name="billing_churn"
    )
)

service_silver, service_quarantine_df = (
    synchronize_domain_quarantine(
        silver_df=service_silver,
        quarantine_df=service_quarantine_df,
        all_quarantine_ids_df=all_quarantine_ids_df,
        domain_name="service_provisioning"
    )
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 84, Finished, Available, Finished, False)

In [76]:
print("Final Silver counts")
print("CRM:", crm_silver.count())
print("Billing:", billing_silver.count())
print("Service:", service_silver.count())

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 85, Finished, Available, Finished, False)

Final Silver counts
CRM: 6857
Billing: 6857
Service: 6857


In [77]:
crm_ids = crm_silver.select("customer_id").distinct()
billing_ids = billing_silver.select("customer_id").distinct()
service_ids = service_silver.select("customer_id").distinct()

print(
    "CRM missing from Billing:",
    crm_ids.join(
        billing_ids,
        on="customer_id",
        how="left_anti"
    ).count()
)

print(
    "Billing missing from CRM:",
    billing_ids.join(
        crm_ids,
        on="customer_id",
        how="left_anti"
    ).count()
)

print(
    "CRM missing from Service:",
    crm_ids.join(
        service_ids,
        on="customer_id",
        how="left_anti"
    ).count()
)

print(
    "Service missing from CRM:",
    service_ids.join(
        crm_ids,
        on="customer_id",
        how="left_anti"
    ).count()
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 86, Finished, Available, Finished, False)

CRM missing from Billing: 0
Billing missing from CRM: 0
CRM missing from Service: 0
Service missing from CRM: 0


In [78]:
print("Final quarantine counts")
print("CRM:", crm_quarantine_df.count())
print("Billing:", billing_quarantine_df.count())
print("Service:", service_quarantine_df.count())

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 87, Finished, Available, Finished, False)

Final quarantine counts
CRM: 186
Billing: 186
Service: 186


In [79]:
print(f"Total nulls in crm: {sum(crm_silver.filter(F.col(c).isNull()).count() for c in crm_silver.columns)}")
print(f"Total nulls in billing: {sum(billing_silver.filter(F.col(c).isNull()).count() for c in billing_silver.columns if c != 'churn_reason')}")
print(f"Total nulls in service: {sum(service_silver.filter(F.col(c).isNull()).count() for c in service_silver.columns)}")

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 88, Finished, Available, Finished, False)

Total nulls in crm: 0
Total nulls in billing: 0
Total nulls in service: 0


# **Validating no data was missed**

In [80]:
def validate_split(source_df, valid_df, quarantine_df, table_name):

    source_count = source_df.count()
    valid_count = valid_df.count()
    quarantine_count = quarantine_df.count()

    print(f"{table_name} source rows     : {source_count}")
    print(f"{table_name} valid rows      : {valid_count}")
    print(f"{table_name} quarantine rows : {quarantine_count}")

    assert source_count == valid_count + quarantine_count, (
        f"{table_name}: source count does not equal "
        "valid count plus quarantine count"
    )

    print(f"{table_name}: split validation passed")

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 90, Finished, Available, Finished, False)

In [81]:
validate_split(
    crm_silver_df,
    crm_silver,
    crm_quarantine_df,
    "CRM"
)

validate_split(
    service_silver_df,
    service_silver,
    service_quarantine_df,
    "Service"
)

validate_split(
    billing_silver_df,
    billing_silver,
    billing_quarantine_df,
    "Billing"
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 91, Finished, Available, Finished, False)

CRM source rows     : 7043
CRM valid rows      : 6857
CRM quarantine rows : 186
CRM: split validation passed
Service source rows     : 7043
Service valid rows      : 6857
Service quarantine rows : 186
Service: split validation passed
Billing source rows     : 7043
Billing valid rows      : 6857
Billing quarantine rows : 186
Billing: split validation passed


# **Check whether customers match across sources**

In [82]:
crm_customers = crm_silver_df.select("customer_id")
service_customers = service_silver_df.select("customer_id")
billing_customers = billing_silver_df.select("customer_id")

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 92, Finished, Available, Finished, False)

In [83]:
crm_missing_service = crm_customers.join(
    service_customers,
    on="customer_id",
    how="left_anti"
)

print(
    "CRM customers missing from Service:",
    crm_missing_service.count()
)

display(crm_missing_service.limit(20))

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 93, Finished, Available, Finished, False)

CRM customers missing from Service: 0


SynapseWidget(Synapse.DataFrame, e733351d-c9a5-4d8c-9628-425f9f9d3726)

In [84]:
crm_missing_billing = crm_customers.join(
    billing_customers,
    on="customer_id",
    how="left_anti"
)

print(
    "CRM customers missing from Billing:",
    crm_missing_billing.count()
)

display(crm_missing_billing.limit(20))

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 94, Finished, Available, Finished, False)

CRM customers missing from Billing: 0


SynapseWidget(Synapse.DataFrame, 5238d0ad-05c2-4059-9b95-12c4e9dc4caa)

In [85]:
service_missing_crm = service_customers.join(
    crm_customers,
    on="customer_id",
    how="left_anti"
)

print(
    "Service customers missing from CRM:",
    service_missing_crm.count()
)

display(service_missing_crm.limit(20))

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 95, Finished, Available, Finished, False)

Service customers missing from CRM: 0


SynapseWidget(Synapse.DataFrame, 4c68e8eb-1637-498c-b017-bcdf4b1cc21a)

# **Saving dataframes to silver tables in the silver lakehouse**

In [86]:
# 1. Fetch properties of your non-default Lakehouse
lakehouse_info = notebookutils.lakehouse.getWithProperties("lh_telecom_silver")

# 2. Extract the base ABFS path from the properties sub-dictionary
base_abfss_path = lakehouse_info['properties']['abfsPath']

# 3. Construct your specific paths
abfss_tables_path = f"{base_abfss_path}/Tables"
abfss_files_path = f"{base_abfss_path}/Files"

print("Tables Path:", abfss_tables_path)
print("Files Path:", abfss_files_path)


StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 96, Finished, Available, Finished, False)

Tables Path: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables
Files Path: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Files


In [87]:
silver_lakehouse_path = ("abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables")

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 97, Finished, Available, Finished, False)

In [88]:
def write_to_silver_lakehouse(
    df,
    table_name,
    silver_lakehouse_path,
    mode="overwrite"
):
    """
    Write a Spark DataFrame as a Delta table to the Silver Lakehouse
    without changing the notebook's default Bronze Lakehouse.
    """

    destination_path = (
        f"{silver_lakehouse_path}/dbo/{table_name}"
    )

    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .save(destination_path)
    )

    print(f"Successfully written: {table_name}")
    print(f"Destination: {destination_path}")

    return destination_path

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 98, Finished, Available, Finished, False)

In [89]:
crm_silver_path = write_to_silver_lakehouse(
    df=crm_silver,
    table_name="silver_crm_customer",
    silver_lakehouse_path=silver_lakehouse_path
)

service_silver_path = write_to_silver_lakehouse(
    df=service_silver,
    table_name="silver_service_provisioning",
    silver_lakehouse_path=silver_lakehouse_path
)

billing_silver_path = write_to_silver_lakehouse(
    df=billing_silver,
    table_name="silver_billing_churn",
    silver_lakehouse_path=silver_lakehouse_path
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 99, Finished, Available, Finished, False)

Successfully written: silver_crm_customer
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/silver_crm_customer
Successfully written: silver_service_provisioning
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/silver_service_provisioning
Successfully written: silver_billing_churn
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/silver_billing_churn


In [90]:
crm_quarantine_path = write_to_silver_lakehouse(
    df=crm_quarantine_df,
    table_name="quarantine_crm_customer",
    silver_lakehouse_path=silver_lakehouse_path
)

service_quarantine_path = write_to_silver_lakehouse(
    df=service_quarantine_df,
    table_name="quarantine_service_provisioning",
    silver_lakehouse_path=silver_lakehouse_path
)

billing_quarantine_path = write_to_silver_lakehouse(
    df=billing_quarantine_df,
    table_name="quarantine_billing_churn",
    silver_lakehouse_path=silver_lakehouse_path
)

StatementMeta(, d475b195-8b1a-4e0e-88d8-2548b1299579, 100, Finished, Available, Finished, False)

Successfully written: quarantine_crm_customer
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/quarantine_crm_customer
Successfully written: quarantine_service_provisioning
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/quarantine_service_provisioning
Successfully written: quarantine_billing_churn
Destination: abfss://b73a3d86-4601-4c6f-8702-64ddd7a30e26@onelake.dfs.fabric.microsoft.com/90507d84-bf3c-4eed-9a96-93f374c56220/Tables/dbo/quarantine_billing_churn
